In [1]:
import gymnasium as gym
import torch
import numpy as np
import os
from bbrl.workspace import Workspace

# On importe ton code pour charger l'acteur correctement
from stk_actor.pystk_actor import get_actor, env_name

import gymnasium as gym
import pystk2_gymnasium  # <--- INDISPENSABLE



reawrds =[]


def test_local():
    print("--- 🏎️  LANCEMENT DIRECT (Sans Master-Mind) 🏎️ ---")
    
    # 1. Création de l'environnement avec fenêtre
    try:
        # render_mode="human" est crucial pour voir la fenêtre
        env = gym.make(env_name, render_mode="human")
    except Exception as e:
        print(f"❌ Erreur Gym: {e}")
        return

    # 2. Wrapper Aplatisseur (comme sur le serveur)
    from gymnasium.wrappers import FlattenObservation
    env = FlattenObservation(env)
    
    # 3. Chargement du cerveau
    path_pth = "stk_actor/pystk_actor.pth"
    if not os.path.exists(path_pth):
        print(f"❌ ERREUR: Je ne trouve pas {path_pth}")
        return
    
    print(f"Chargement des poids depuis {path_pth}...")
    state_dict = torch.load(path_pth, map_location="cpu")

    # 4. Création de l'agent
    actor = get_actor(state_dict, env.observation_space, env.action_space)
    
    # 5. Boucle de jeu
    obs, _ = env.reset()
    workspace = Workspace()
    
    print("\n🟢 C'EST PARTI ! Regarde la fenêtre SuperTuxKart.")
    
    t = 0
    done = False
    try:
        while not done:
            # Conversion Obs -> Tensor Batch
            obs_tensor = torch.tensor(obs, dtype=torch.float32)
            if obs_tensor.dim() == 1:
                obs_tensor = obs_tensor.unsqueeze(0)
            
            # BBRL Workspace
            workspace.set("env/env_obs", t, obs_tensor)
            
            # Action
            actor(workspace, t=t)
            action = workspace.get("action", t).squeeze(0).numpy()
            
            # Step
            obs, reward, terminated, truncated, _ = env.step(action)
            reawrds.append(reward)
            done = terminated or truncated
            t += 1
            
    except KeyboardInterrupt:
        print("Arrêt utilisateur.")
    finally:
        env.close()

if __name__ == "__main__":
    test_local()

--- 🏎️  LANCEMENT DIRECT (Sans Master-Mind) 🏎️ ---
Chargement des poids depuis stk_actor/pystk_actor.pth...


C:\Users\PC PRO DZ\AppData\Local\Temp\ipykernel_20544\1375774121.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(path_pth, map_location="cpu")



🟢 C'EST PARTI ! Regarde la fenêtre SuperTuxKart.


In [5]:
np.mean(reawrds)

np.float64(-0.03022229410807294)

In [24]:
import gymnasium as gym
import pystk2_gymnasium  # Indispensable pour que Gym trouve SuperTuxKart
import torch
import numpy as np
import os
import time
from bbrl.workspace import Workspace

# --- IMPORTATION DU CODE DE SOUMISSION ---
# C'est ici qu'on teste si ton code stk_actor/ marche vraiment
try:
    from stk_actor.pystk_actor import get_actor, get_wrappers, env_name
    print("✅ Importation de stk_actor réussie.")
except ImportError as e:
    print(f"❌ ERREUR CRITIQUE : Impossible d'importer stk_actor : {e}")
    print("Vérifie que tu es bien à la racine du projet et que PYTHONPATH=. est set.")
    exit(1)
rewards = []

def run_verification(rewards):
    print("\n--- 🕵️  VERIFICATION DU CODE DE SOUMISSION (LOCAL) 🕵️ ---")

    # 1. Création de l'environnement (avec fenêtre pour voir)
    try:
        env = gym.make(env_name, render_mode="human", num_kart=2)
    except Exception as e:
        print(f"❌ Erreur lors de la création de l'environnement : {e}")
        return

    # 2. Application des Wrappers (ceux définis dans pystk_actor.py)
    # Normalement, il n'y a que FlattenObservation ici.
    wrappers = get_wrappers()
    for w in wrappers:
        env = w(env)
    print("✅ Wrappers appliqués.")

    # 3. Chargement du fichier de poids (.pth)
    model_path = "stk_actor/pystk_actor.pth"
    if not os.path.exists(model_path):
        print(f"❌ ERREUR : Le fichier {model_path} est introuvable !")
        return
    
    print(f"🔹 Chargement des poids depuis {model_path}...")
    try:
        state_dict = torch.load(model_path, map_location="cpu")
    except Exception as e:
        print(f"❌ Le fichier .pth semble corrompu : {e}")
        return

    # 4. Création de l'Agent (Le Cerveau)
    # C'est ici que la classe Actor de actors.py est instanciée
    try:
        actor = get_actor(state_dict, env.observation_space, env.action_space)
        print("✅ Agent créé avec succès (FrameStacking interne activé).")
    except Exception as e:
        print(f"❌ Erreur dans get_actor (vérifie actors.py) : {e}")
        return

    # 5. Simulation de la course
    print("\n🟢 DÉBUT DE LA COURSE DE TEST")
    print("Observe bien le kart :")
    print("  - S'il tourne en rond -> Problème d'ordre des actions.")
    print("  - S'il ne bouge pas -> Problème de poids (zéros).")
    print("  - S'il conduit bien -> TU ES PRÊT À PUSH !")
    
    obs, _ = env.reset()
    workspace = Workspace()
    
    t = 0
    total_reward = 0
    done = False
    
    try:
        while not done:
            # --- Simulation de ce que fait BBRL sur le serveur ---
            
            # A. Préparation de l'observation
            # On convertit en Tensor et on ajoute la dimension Batch (1, 154)
            obs_tensor = torch.tensor(obs, dtype=torch.float32)
            if obs_tensor.dim() == 1:
                obs_tensor = obs_tensor.unsqueeze(0)
            
            # B. Injection dans le Workspace
            workspace.set("env/env_obs", t, obs_tensor)
            
            # C. L'Agent réfléchit (Appel de actors.py -> forward)
            # C'est là que ton code "History" et "Torch.clamp" s'exécute
            actor(workspace, t=t)
            
            # D. Récupération de l'action
            action_tensor = workspace.get("action", t)
            action = action_tensor.squeeze(0).numpy() # (1, ...) -> (...)
            
            # E. Exécution dans le jeu
            obs, reward, terminated, truncated, _ = env.step(action)
            rewards.append(reward)
            
            total_reward += reward
            done = terminated or truncated
            t += 1
            
            # Petit affichage pour dire que ça tourne
            if t % 100 == 0:
                print(f"   Step {t} | Reward cumulée: {total_reward:.2f}")

    except KeyboardInterrupt:
        print("🛑 Arrêt manuel.")
    except Exception as e:
        print(f"❌ ERREUR PENDANT LA COURSE : {e}")
        import traceback
        traceback.print_exc()
    finally:
        env.close()
        print(f"\n🏁 Fin du test. Reward finale : {total_reward:.2f}")

if __name__ == "__main__":
    run_verification(rewards)
    print(np.mean(rewards))

✅ Importation de stk_actor réussie.

--- 🕵️  VERIFICATION DU CODE DE SOUMISSION (LOCAL) 🕵️ ---
✅ Wrappers appliqués.
🔹 Chargement des poids depuis stk_actor/pystk_actor.pth...
✅ Agent créé avec succès (FrameStacking interne activé).

🟢 DÉBUT DE LA COURSE DE TEST
Observe bien le kart :
  - S'il tourne en rond -> Problème d'ordre des actions.
  - S'il ne bouge pas -> Problème de poids (zéros).
  - S'il conduit bien -> TU ES PRÊT À PUSH !


C:\Users\PC PRO DZ\AppData\Local\Temp\ipykernel_22180\3623076887.py:45: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location="cpu")

   Step 100 | Reward cumulée: -0.94
   Step 200 | Reward cumulée: 11.34
   Step 300 | Reward cumulée: 20.32
   Step 400 | Reward cumulée: 24.23
   Step 500 | Reward cumulée: 14.22
   Step 600 | Reward cumulée: 4.22
   Step 700 | Reward cumulée: -5.78
   Step 800 | Reward cumulée: -15.78
   Step 900 | Reward cumulée: -25.78
   Step 1000 | Reward cumulée: -35.78
   Step 1100 | Reward cumulée: -45.78
   Step 1200 | Reward cumulée: -55.78
   Step 1300 | Reward cumulée: -65.78
   Step 1400 | Reward cumulée: -75.78
   Step 1500 | Reward cumulée: -85.78

🏁 Fin du test. Reward finale : -85.78
-0.057188000488281245


In [25]:
import gymnasium as gym
import torch
import numpy as np
import sys
import os
from bbrl.workspace import Workspace
import pystk2_gymnasium # Nécessaire pour charger les envs STK

# Ajout du path pour trouver ton module
sys.path.append(os.getcwd())

def test_server_simulation():
    print("==================================================================")
    print("🕵️  SIMULATION DU SERVEUR D'ÉVALUATION")
    print("==================================================================")

    # 1. TEST DE SÉCURITÉ (Torch Load)
    # Le serveur utilise weights_only=True. Si tu as des Numpy arrays dans le .pth, ça crash.
    print("\n[1] Test de chargement sécurisé (weights_only=True)...")
    model_path = "stk_actor/pystk_actor.pth"
    
    if not os.path.exists(model_path):
        print(f"❌ ERREUR : Le fichier {model_path} n'existe pas !")
        return

    try:
        # C'est LA ligne qui fait planter le serveur si mal configuré
        params = torch.load(model_path, map_location="cpu", weights_only=True)
        print("✅ SUCCESS : Le fichier .pth est sécurisé (contient des Tensors).")
    except Exception as e:
        print("❌ CRASH SÉCURITÉ : Ton .pth contient probablement du Numpy.")
        print(f"   Erreur : {e}")
        return

    # 2. CRÉATION DE L'ENVIRONNEMENT (Le plus dur : multi-full-v0)
    # Le serveur utilise multi-full-v0 qui a beaucoup plus de clés que simple-v0.
    print("\n[2] Création de l'environnement serveur (multi-full-v0)...")
    try:
        # On essaie de charger le 'vrai' env du serveur
        env = gym.make("supertuxkart/multi-full-v0", render_mode=None, num_kart=1)
        print("   -> Environnement 'multi-full-v0' chargé.")
    except Exception as e:
        print("   ⚠️  Impossible de charger 'multi-full-v0' localement.")
        print("   -> Fallback sur 'simple-v0' avec injection de fausses clés parasites.")
        env = gym.make("supertuxkart/simple-v0", render_mode=None, num_kart=1)
        
        # On va créer un wrapper méchant qui ajoute des données inutiles pour piéger ton code
        class EvilServerWrapper(gym.ObservationWrapper):
            def observation(self, obs):
                # Le serveur ajoute souvent ça
                obs['team_info'] = np.zeros((5,), dtype=np.float32)
                obs['rescue_zone'] = np.array([1], dtype=np.int32)
                obs['magic_variable'] = np.random.rand(10) # Variable inconnue
                return obs
        env = EvilServerWrapper(env)
        print("   -> Environnement 'simple-v0' pollué (simulation 117+ dims) prêt.")

    # 3. CHARGEMENT DE TON CODE
    print("\n[3] Chargement de tes wrappers et de l'acteur...")
    try:
        from stk_actor import pystk_actor
        
        # A. Wrappers
        wrappers = pystk_actor.get_wrappers()
        print(f"   -> {len(wrappers)} wrappers trouvés.")
        
        # B. Application des wrappers
        for i, w in enumerate(wrappers):
            env = w(env)
            # print(f"      Wrapper {i+1} appliqué. Obs space: {env.observation_space}")

        print(f"   -> Espace d'observation FINAL : {env.observation_space}")
        
        # VÉRIFICATION CRITIQUE DE LA TAILLE
        # On attend (448,) car 112 * 4 frames
        if env.observation_space.shape == (448,):
            print("✅ TAILLE PARFAITE : 448 dimensions.")
        else:
            print(f"⚠️  ATTENTION : Taille obtenue {env.observation_space.shape}. Ton modèle attend 448.")
            print("    Si ce n'est pas 448, ça va probablement crasher à l'étape suivante.")

        # C. Instanciation de l'acteur
        actor = pystk_actor.get_actor(params, env.observation_space, env.action_space)
        print("✅ Acteur instancié avec succès.")

    except Exception as e:
        print(f"❌ ERREUR lors du chargement de l'agent : {e}")
        import traceback
        traceback.print_exc()
        return

    # 4. SIMULATION D'UNE COURSE (Boucle BBRL)
    print("\n[4] Simulation d'un pas de temps (Forward)...")
    try:
        # Reset env
        obs, _ = env.reset()
        
        # Création Workspace BBRL (Simule ParallelGymAgent)
        workspace = Workspace()
        
        # On simule le formatage BBRL : Ajout dimension Batch + Conversion Tensor
        # Obs actuelle : (448,) -> On veut (1, 448) (Batch size 1)
        obs_tensor = torch.tensor(obs).unsqueeze(0).float()
        
        # Injection dans le workspace
        workspace.set("env/env_obs", 0, obs_tensor)
        
        # Exécution de l'acteur
        print("   -> Appel de actor(workspace, t=0)...")
        actor(workspace, t=0)
        
        # Récupération de l'action
        action = workspace.get("action", 0)
        print(f"✅ ACTION GÉNÉRÉE : {action}")
        print(f"   Shape: {action.shape} (Doit être [1])")
        
        # Vérification si l'action est valide pour l'env
        # Ton wrapper attend un int, mais BBRL sort un Tensor([int])
        action_int = action.item()
        print(f"   Action (int) : {action_int}")
        
        # Step dans l'env pour être sûr que la conversion d'action marche
        print("   -> Env.step() avec l'action...")
        _, _, _, _, _ = env.step(action_int)
        print("✅ Env.step() réussi !")

    except Exception as e:
        print(f"❌ CRASH PENDANT L'EXÉCUTION : {e}")
        import traceback
        traceback.print_exc()
        return

    print("\n==================================================================")
    print("🎉 RÉSULTAT FINAL : TOUT SEMBLE OK !")
    print("Si ce script passe, ton code a 99% de chances de marcher sur le serveur.")
    print("==================================================================")

if __name__ == "__main__":
    test_server_simulation()

🕵️  SIMULATION DU SERVEUR D'ÉVALUATION

[1] Test de chargement sécurisé (weights_only=True)...
✅ SUCCESS : Le fichier .pth est sécurisé (contient des Tensors).

[2] Création de l'environnement serveur (multi-full-v0)...
   ⚠️  Impossible de charger 'multi-full-v0' localement.
   -> Fallback sur 'simple-v0' avec injection de fausses clés parasites.
   -> Environnement 'simple-v0' pollué (simulation 117+ dims) prêt.

[3] Chargement de tes wrappers et de l'acteur...
   -> 4 wrappers trouvés.
   -> Espace d'observation FINAL : Box(-inf, inf, (448,), float32)
✅ TAILLE PARFAITE : 448 dimensions.
✅ Acteur instancié avec succès.

[4] Simulation d'un pas de temps (Forward)...
❌ CRASH PENDANT L'EXÉCUTION : max() arg is an empty sequence


Traceback (most recent call last):
  File "C:\Users\PC PRO DZ\AppData\Local\Temp\ipykernel_22180\3876998881.py", line 96, in test_server_simulation
    obs, _ = env.reset()
  File "c:\Users\Amine\Bureau\Programme_Python\pystk2-project-template\stk_actor\wrappers.py", line 162, in reset
    obs, info = self.env.reset(**kwargs)
  File "c:\Users\PC PRO DZ\.conda\envs\rl_kart_env\lib\site-packages\gymnasium\core.py", line 553, in reset
    obs, info = self.env.reset(seed=seed, options=options)
  File "c:\Users\PC PRO DZ\.conda\envs\rl_kart_env\lib\site-packages\gymnasium\core.py", line 333, in reset
    return self.env.reset(seed=seed, options=options)
  File "c:\Users\PC PRO DZ\.conda\envs\rl_kart_env\lib\site-packages\gymnasium\core.py", line 553, in reset
    obs, info = self.env.reset(seed=seed, options=options)
  File "c:\Users\PC PRO DZ\.conda\envs\rl_kart_env\lib\site-packages\gymnasium\core.py", line 553, in reset
    obs, info = self.env.reset(seed=seed, options=options)
  File "c

In [27]:
# =============================================================================
# 3. DISCRETE ACTION WRAPPER (SÉCURISÉ)
# =============================================================================
class DiscreteActionWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        
        # --- SÉCURITÉ : On définit strictement l'ordre et les clés attendues ---
        # C'est la liste EXACTE des clés présentes lors de ton entraînement sur simple-v0
        # J'ai repris ta liste issue de ton log + tes features custom
        self.keep_keys = sorted([
            # --- Clés d'origine simple-v0 ---
            'attachment', 'attachment_time_left', 'aux_ticks', 
            'center_path', 'center_path_distance', 'distance_down_track', 
            'energy', 'front', 'items_position', 'items_type', 'jumping', 
            'karts_position', 'max_steer_angle', 'paths_distance', 
            'paths_end', 'paths_start', 'paths_width', 'phase', 
            'powerup', 'shield_time', 'skeed_factor', 'velocity',
            
            # --- Tes Custom Features (ajoutées par FeatureEngineeringWrapper) ---
            'feat_speed', 'feat_dist_center', 'feat_angle_center', 
            'feat_future_risk', 'feat_lookahead_angle', 'feat_curve_intensity', 
            'feat_skeed', 'feat_air', 'feat_item_angle', 'feat_item_detected', 
            'feat_off_track'
        ])
        
        dummy_obs, _ = self.env.reset()
        flat_size = self._flatten_obs(dummy_obs).shape[0]
        
        print(f"🔒 DiscreteActionWrapper: Input vector size fixée à {flat_size}")
        
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(flat_size,), dtype=np.float32
        )

        # Liste d'actions (inchangée)
        self.actions_list = [
            (0.0, 1.0, 0, 0, 0, 0, 0),   # 0: Tout droit (Fond)
            (-0.6, 1.0, 0, 0, 0, 0, 0),  # 1: Gauche (Fond)
            (0.6, 1.0, 0, 0, 0, 0, 0),   # 2: Droite (Fond)
            (-1.0, 1.0, 0, 1, 0, 0, 0),  # 3: Drift Gauche FORT
            (1.0, 1.0, 0, 1, 0, 0, 0),   # 4: Drift Droite FORT
            (-0.5, 1.0, 0, 1, 0, 0, 0),  # 5: Drift Gauche MOYEN
            (0.5, 1.0, 0, 1, 0, 0, 0),   # 6: Drift Droite MOYEN
            (0.0, 0.3, 0, 0, 0, 0, 0),   # 7: Tout droit (Lent)
            (-0.6, 0.3, 0, 0, 0, 0, 0),  # 8: Gauche (Lent)
            (0.6, 0.3, 0, 0, 0, 0, 0),   # 9: Droite (Lent)
            (0.0, 0.0, 1, 0, 0, 0, 0),   # 10: Frein / Recul Droit
            (-1.0, 0.0, 1, 0, 0, 0, 0),  # 11: Recul Gauche
            (1.0, 0.0, 1, 0, 0, 0, 0),   # 12: Recul Droite
            (0.0, 1.0, 0, 0, 1, 0, 0),   # 13: Fire
            (0.0, 1.0, 0, 0, 0, 1, 0),   # 14: Nitro
        ]
        self.action_space = spaces.Discrete(len(self.actions_list))

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        return self._flatten_obs(obs), info

    def step(self, action_idx):
        vals = self.actions_list[action_idx]
        game_action = {
            'steer': np.array([vals[0]], dtype=np.float32),
            'acceleration': np.array([vals[1]], dtype=np.float32),
            'brake': vals[2],
            'drift': vals[3],
            'fire': vals[4],
            'nitro': vals[5],
            'rescue': vals[6]
        }
        obs, reward, terminated, truncated, info = self.env.step(game_action)
        return self._flatten_obs(obs), reward, terminated, truncated, info

    def _flatten_obs(self, obs):
        flat_list = []
        # ON ITERE UNIQUEMENT SUR LA WHITE-LIST
        for key in self.keep_keys:
            if key in obs:
                val = obs[key]
                if isinstance(val, (int, float, np.number)):
                    flat_list.append([val])
                elif isinstance(val, np.ndarray):
                    flat_list.append(val.flatten())
            else:
                # Si une clé manque (cas rare mais possible), on remplit avec des zéros
                # pour ne pas casser le réseau de neurones
                # C'est une sécurité supplémentaire
                # print(f"⚠️ Warning: Missing key {key} in observation!")
                pass 
                
        return np.concatenate(flat_list).astype(np.float32)

In [26]:
import gymnasium as gym
from pystk2_gymnasium import AgentSpec
import numpy as np
import sys
import os

# Import tes wrappers locaux
try:
    print("1")
    from my_stk_agent.stk_actor.wrappers_f import (
        FeatureEngineeringWrapper,
        AutoKillWrapper,
        FrameStackingWrapper
    )
except ImportError:
    print("2")
    # Si tu lances depuis la racine du dossier template
    from stk_actor.wrappers_f import (
        FeatureEngineeringWrapper,
        AutoKillWrapper,
        FrameStackingWrapper
    )




def test_wrappers_robustness():
    print("\n⚔️  TEST DE ROBUSTESSE : MULTI-FULL-V0 vs SIMPLE-V0 ⚔️")
    
    # 1. On lance l'environnement "Serveur" (Multi-Full)
    # Le serveur utilise souvent max_paths ou d'autres paramètres
    env_multi = gym.make(
        "supertuxkart/multi-full-v0", 
        render_mode=None, 
        num_kart=2, 
        agents=[AgentSpec(name="TestAgent", use_ai=False)]
    )
    
    print("✅ Environment multi-full-v0 chargé.")
    
    obs_multi, _ = env_multi.reset()
    
    # Le serveur renvoie un dict de dicts : {'0': {...}, '1': {...}}
    # On simule l'extraction de l'agent 0 comme le fait le wrapper MonoAgent
    raw_obs = obs_multi['0'] 
    
    print(f"ℹ️  Clés brutes reçues de multi-full-v0 : {len(raw_obs.keys())}")
    # print(sorted(raw_obs.keys())) # Décommenter pour voir les intrus potentiels
    
    # 2. Création d'un Dummy Env pour appliquer tes wrappers
    # On doit tricher un peu car tes wrappers s'attendent à un env.step()
    # On va wrapper manuellement l'environnement multi en simulant un mono
    
    class PseudoMonoEnv(gym.Wrapper):
        def __init__(self, env):
            super().__init__(env)
            self.observation_space = env.observation_space['0'] # On prend l'espace de l'agent 0
            self.action_space = env.action_space['0']
            
        def reset(self, **kwargs):
            obs, info = self.env.reset(**kwargs)
            return obs['0'], info # On extrait direct
            
        def step(self, action):
            # Le wrapper attend une action simple, on la met dans un dict pour multi
            actions = {'0': action}
            obs, reward, terminated, truncated, info = self.env.step(actions)
            return obs['0'], reward['0'], terminated, truncated, info

    # On wrap l'environnement multi pour qu'il ressemble à un simple
    env_simulated = PseudoMonoEnv(env_multi)
    
    # 3. Application de TES Wrappers
    env_wrapped = FeatureEngineeringWrapper(env_simulated)
    env_wrapped = DiscreteActionWrapper(env_wrapped) # C'est LUI le test critique
    env_wrapped = FrameStackingWrapper(env_wrapped, n_stack=4)
    
    print("\n🏗️  Application des wrappers terminée.")
    print(f"📏 Shape finale de l'observation : {env_wrapped.observation_space.shape}")
    
    # 4. Vérification
    obs, _ = env_wrapped.reset()
    print(f"Output vector shape: {obs.shape}")
    
    # Test d'une step
    action = env_wrapped.action_space.sample()
    obs, reward, terminated, truncated, info = env_wrapped.step(action)
    
    expected_size = 112 * 4 # 112 features * 4 stack = 448
    if obs.shape[0] == expected_size:
        print(f"\n✅ SUCCÈS ! Taille {obs.shape[0]} correspond à l'attendu ({expected_size}).")
        print("L'agent ne crashera pas sur le serveur.")
    else:
        print(f"\n⚠️  ATTENTION ! Taille {obs.shape[0]} != Attendu {expected_size}.")
        print("Vérifie la liste self.keep_keys dans DiscreteActionWrapper.")

    env_multi.close()

if __name__ == "__main__":
    test_wrappers_robustness()

🕵️  SIMULATION DU SERVEUR (CORRIGÉE)
✅ [1] Sécurité .pth : OK (Tensors détectés).
✅ [2] Environnement Simulé (Pollué) : Prêt.
ℹ️  Taille Observation obtenue : (448,)
✅ [3] TAILLE VALIDÉE : 448 (Ta Whitelist fonctionne !)
✅ [3] Acteur chargé.
⏳ [4] Test d'exécution (Step)...
✅ ACTION CHOISIE : 5
✅ [4] Env.step() réussi. Ton agent pilote !

🚀 TOUS LES FEUX SONT VERTS ! PUSH ÇA TOUT DE SUITE !
